**The Grade Level Appropriateness Evaluator** helps you assess whether AI-generated text is suitable for a given grade band. When you run a passage through the evaluator, it returns a structured output that includes:

* Grade: The target grade band where the text can be read independently.

* Alternative_grade: A fallback grade band where the text may still be useful with extra support.

* Scaffolding_needed: Specific supports (like vocabulary pre-teaching) that make the text accessible outside the target band.

This gives you  a clear signal about text difficulty and how it can be adapted, so you can generate content that matches learners’ needs.


### Install required python packages

In [ ]:
#Install relevant packages
%pip install -qU pydantic dotenv langchain langchain_google_genai

In [ ]:
#Load necessary packages
import getpass
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts.chat import HumanMessagePromptTemplate
from langchain_core.messages import SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

### Preparing prompt 

In [ ]:
#This is the system prompt, user prompt and model output setting. Do not change this
from prompts.gla_prompts import gla_system_prompt, gla_user_prompt

class OutputRanges(BaseModel):
    reasoning: str = Field(description="your reasoning for your answer in numbered bullet points for 4 steps with a 5th bullet point for synthesis.")
    grade: str = Field(description="the appropriate grade level for the text")
    alternative_grade: str = Field(description="an alternative grade level for the text")
    scaffolding_needed: str = Field(description="scaffolding needed for the text to be appropriate for the alternative grade")

prompt_option = {
       "promptName": "anet_steps_fk_anet_feedback_refined_no_sb",
        "system": gla_system_prompt,
        "user": gla_user_prompt,
        "inputVars": ["text"],
        "outputParser": JsonOutputParser(pydantic_object=OutputRanges),
    }

### Set up the Vocabulary Evaluator function

In [ ]:
# Check for the API key
load_dotenv()
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google AI API key: ")

#Step 1: prepare the LLM model invocation with the text you want to evaluate
def run_gla_evaluator(text):

    #Construct data for LLM invocation
    input_data = {
        "text": text
    }

    #Set up model to use
    model_name = 'gemini-2.5-pro'

    llm = ChatGoogleGenerativeAI(model=model_name, temperature=0.25, timeout=120000)
    messages= [
        SystemMessage(content=prompt_option["system"]),
        HumanMessagePromptTemplate.from_template(prompt_option["user"])
    ]

    #Prepare prompt
    prompt = ChatPromptTemplate(
        messages,
        input_variables=prompt_option["inputVars"],
        partial_variables={
            "format_instructions": prompt_option["outputParser"].get_format_instructions()
        }
    )
    #Invoke the chain
    chain = prompt | llm | JsonOutputParser()

    #return output
    output = chain.invoke(input_data)
    print(
        f"""
            Target grade band is: {output['grade']}\n
            Alternative grade band is: {output['alternative_grade']}\n
            Recommended scaffolding for alternative grade band is: {output['scaffolding_needed']}\n
            Reasoning:\n{output['reasoning']}\n
        """
    )
    return output


# Try evaluating text to determine appropriate grade level

Evaluator may take up to a minute to run. While the evaluator is running, you will see that the status of the kernel is busy. For Jupyter, you can see the status of kernel at the bottom of the page.

To evaluate your own text, replace the text and run the cell again.

In [ ]:
#Swap out text here for your own text to evaluate
text = """
Tides are the rise and fall of sea levels caused by the combined effects of the gravitational forces exerted by the Moon and the Sun and the rotation of the Earth.
The times and amplitude of tides at a locale are influenced by the alignment of the Sun and Moon, by the pattern of tides in the deep ocean, by the amphidromic systems of the oceans, and the shape of the coastline and near-shore bathymetry (see Timing). Some shorelines experience a semi-diurnal tide - two nearly equal high and low tides each day. Other locations experience a diurnal tide - only one high and low tide each day. A "mixed tide"; two uneven tides a day, or one high and one low, is also possible.
Tides vary on timescales ranging from hours to years due to a number of factors. To make accurate records, tide gauges at fixed stations measure the water level over time. Gauges ignore variations caused by waves with periods shorter than minutes. These data are compared to the reference (or datum) level usually called mean sea level.
"""

output = run_gla_evaluator(text)